In [1]:
import pandas as pd

df = pd.read_csv('../outputs/carolinian_combined.tsv', sep='\t', dtype=str, keep_default_na=False)
print(f'Loaded {len(df)} rows')
df.head(3)

Loaded 8145 rows


,Block_ID,Entry_Type,Headword,Parent_Lexeme,Sense_Number,Homonym_Number,Gloss,Gloss_Secondary,Usage_Note,POS,...,Cross_References,Examples,Example_Glosses,Source_Page,Morphemic_Representation,Etymology,Variant_Form,Dialect,Inflection,Notes
0,aa,main,aa,,,,Name of the first letter of the Carolinian alp...,,,n,...,,,,page_33,,,,,,
1,a–:1,main,a–,,,1,Prefixed to nouns and verbs to derive transiti...,,,caus pref,...,Related akka– 2,Amwongoow | Ammasagh | Aita | Abwiibwi,"to cause to eat, to feed | to cause fear, be f...",page_33,A-,[POC *(pa)ka],,,,
2,a–:2,main,a–,,,2,Prefixed to cardinal numbers; co-occurs with -...,,,ordinal pref,...,Compare a– 1,Afaamalúl | Alimoowal | Aruufóschol,fourth person or animal | fifth general object...,page_33,A-,[POC *ka],,,,


## Check 1: Headword is not empty

In [2]:
flagged = df[df['Headword'].str.strip() == '']
print(f'Flagged: {len(flagged)} rows with empty headword')
if len(flagged):
    display(flagged[['Source_Page', 'Entry_Type', 'Headword', 'Gloss']])

Flagged: 0 rows with empty headword


## Check 2: Gloss is not empty (allow if Variant_Form is set)

In [3]:
empty_gloss = df[df['Gloss'].str.strip() == '']
# Allow entries where Variant_Form is filled (these are redirect entries)
flagged = empty_gloss[empty_gloss['Variant_Form'].str.strip() == '']
print(f'Flagged: {len(flagged)} rows with empty gloss and no variant form')
if len(flagged):
    display(flagged[['Source_Page', 'Entry_Type', 'Headword', 'Gloss', 'Variant_Form', 'Cross_References']])

Flagged: 86 rows with empty gloss and no variant form


,Source_Page,Entry_Type,Headword,Gloss,Variant_Form,Cross_References
57,page_37,main,abwóppw,,,
64,page_37,main,abwungubwung,,,
959,page_90,main,aseew,,,Variant of asééw
967,page_90,main,assita,,,Variant of assiti
1247,page_108,main,ayúniyar,,,Variant of nayúniyar
...,...,...,...,...,...,...
7460,page_453,main,túlungasch,,,"Related ngáásch | Synonym túlaaw, tuumw"
7476,page_454,main,tchamaaw,,,Variant of tchemaaw
7618,page_462,main,uri,,,Variant of weri
7624,page_462,main,uruló,,,Variant of uroló


## Check 3: Subentry must have a non-empty Parent_Lexeme

In [4]:
subentries = df[df['Entry_Type'] == 'subentry']
flagged = subentries[subentries['Parent_Lexeme'].str.strip() == '']
print(f'Flagged: {len(flagged)} subentries with empty Parent_Lexeme')
if len(flagged):
    display(flagged[['Source_Page', 'Headword', 'Parent_Lexeme', 'Gloss']])

Flagged: 0 subentries with empty Parent_Lexeme


## Check 4: Duplicate headwords must have Sense_Number or Homonym_Number

In [5]:
dupes = df[df.duplicated(subset=['Headword', 'Entry_Type'], keep=False)]
flagged = dupes[
    (dupes['Sense_Number'].str.strip() == '') &
    (dupes['Homonym_Number'].str.strip() == '')
]
print(f'Flagged: {len(flagged)} duplicate headwords (within same Entry_Type) missing both Sense_Number and Homonym_Number')
if len(flagged):
    display(flagged[['Source_Page', 'Entry_Type', 'Headword', 'Sense_Number', 'Homonym_Number', 'Gloss']])

Flagged: 38 duplicate headwords (within same Entry_Type) missing both Sense_Number and Homonym_Number


,Source_Page,Entry_Type,Headword,Sense_Number,Homonym_Number,Gloss
0,page_33,main,aa,,,Name of the first letter of the Carolinian alp...
9,page_34,main,aa,,,Someone or something has done or did an action...
241,page_47,main,ailúúl,,,Variant of ayúlúúl2
367,page_54,main,aláng,,,Variant of áláng
374,page_55,main,aláng,,,Variant of áláng
486,page_61,main,amálimál,,,Variant of amalamal
588,page_68,main,ammwel,,,Variant of ámmwel
1266,page_110,main,áál,,,Variant of yáál 2
1531,page_126,main,bwii,,,Name of the fourth letter of the Carolinian al...
1702,page_135,main,bwii,,,"Sibling of either sex, sister or brother; trad..."


## Check 5: POS is not empty and uses only known tags

In [6]:
VALID_POS = {
    'adv', 'asp', 'caus', 'cl', 'conj', 'dem', 'dir suf', 'idiom', 'intj',
    'n', 'neg', 'num', 'num cl', 'obj suf', 'ord pref', 'poss cl', 'poss suf',
    'pref', 'pron', 'ques', 'redup', 'rel n', 'subj', 'suf', 'tns-asp',
    'v', 'vi', 'vt',
    # Common combinations
    'vi caus', 'vt caus', 'n caus', 'poss pron', 'n pref',
}

# Exclude redirect entries — these legitimately have no POS
is_variant = (
    df['Gloss'].str.strip().str.startswith('Variant of') |
    ((df['Variant_Form'].str.strip() != '') & (df['Gloss'].str.strip() == ''))
)
df_pos = df[~is_variant]

# Check empty POS
empty_pos = df_pos[df_pos['POS'].str.strip() == '']
print(f'Flagged: {len(empty_pos)} rows with empty POS (excluding variant/redirect entries)')
if len(empty_pos):
    display(empty_pos[['Source_Page', 'Entry_Type', 'Headword', 'POS', 'Gloss']].head(20))

# Check unknown POS tags
non_empty_pos = df_pos[df_pos['POS'].str.strip() != '']
unknown = non_empty_pos[~non_empty_pos['POS'].str.strip().isin(VALID_POS)]
print(f'\nFlagged: {len(unknown)} rows with unrecognised POS')
if len(unknown):
    display(unknown[['Source_Page', 'Entry_Type', 'Headword', 'POS']].drop_duplicates('POS'))
    print('Unique unrecognised POS values:', sorted(unknown['POS'].str.strip().unique().tolist()))


Flagged: 88 rows with empty POS (excluding variant/redirect entries)


,Source_Page,Entry_Type,Headword,POS,Gloss
155,page_42,main,aguuwa,,Needle
484,page_61,main,amaleeset,,Sp. of tall tree found on sandy beaches
635,page_70,main,amwooy,,To be very deep (of a wound)
959,page_90,main,aseew,,
967,page_90,main,assita,,
1247,page_108,main,ayúniyar,,
1522,page_125,main,bileed,,
1530,page_125,main,bleet,,
1536,page_126,main,bwaikkér,,
1537,page_126,main,bwaikkérew,,



Flagged: 374 rows with unrecognised POS


,Source_Page,Entry_Type,Headword,POS
1,page_33,main,a–,caus pref
2,page_33,main,a–,ordinal pref
9,page_34,main,aa,pron + asp
20,page_34,main,abwappw,"n, vi caus"
26,page_35,main,abwaat,"n caus, vi caus"
...,...,...,...,...
6408,page_397,main,sáril,Variant of sárel
6410,page_397,main,ssát,Variant of sset
6480,page_401,subentry,-sset,numcl
6614,page_407,main,-ssóbw,neg tnsasp


Unique unrecognised POS values: ['(LN) n', '(LN) vi', '(LN) vt', '(LN) vt suf', '(S) poss', '(S) pron', '(TAN) vt', '(s) poss suf', '(s) vt', 'Poss suf', 'Variant of', 'Variant of abwappw', 'Variant of abwungobwung', 'Variant of faingi', 'Variant of falangáli', 'Variant of fiisigha', 'Variant of fisabwúghúw', 'Variant of fisangaras', 'Variant of fisiigh', 'Variant of fisufósch', 'Variant of fisú-', 'Variant of fááiset', 'Variant of fáálippwel', 'Variant of fáálipó', 'Variant of fáályáp', 'Variant of lauti', 'Variant of lawulaw', 'Variant of pééli', 'Variant of sset', 'Variant of sárel', 'adv suf', 'caus pref', 'conj, comp', 'dem pron', 'dem pron ques', 'dem ques', 'foc', 'intj neg', 'loc', 'loc vt', 'n (idiom)', 'n (idiom?)', 'n (pass)', 'n (redup)', 'n caus, vi caus', 'n phr', 'n ques', 'n, v', 'n, vi', 'n, vi caus', 'neg asp', 'neg tnsasp', 'numcl', 'ordinal pref', 'poss', 'poss cl, n', 'prep', 'pron + asp', 'pron + neg', 'pron + tns-asp', 'prt', 'ques n', 'ques pron', 'v phr', 'v pr

## Check 6: If Examples is not empty, Example_Glosses must not be empty

In [7]:
has_example = df[df['Examples'].str.strip() != '']
flagged = has_example[has_example['Example_Glosses'].str.strip() == '']
print(f'Flagged: {len(flagged)} rows with examples but no example gloss')
if len(flagged):
    display(flagged[['Source_Page', 'Headword', 'Examples', 'Example_Glosses']])

Flagged: 6 rows with examples but no example gloss


,Source_Page,Headword,Examples,Example_Glosses
1025,page_94,aschey,Ascheyiito – throw it here. Ascheyiiló throw i...,
1564,page_128,Bwaragha,Re-Bwaragha,
1816,page_140,bwóór,Bwórol – his coffin,
4386,page_285,malaw,Malaweer Refaluwasch – Carolinian way of life,
5726,page_359,-pé,Epé one container | Ruwapé two containers | El...,
6657,page_410,Sóubwolowat,Re–Sóubwolowat,


## Polishing

### Deduplication: remove cross-page duplicate entries

If the same entry appears on multiple pages (cross-page continuation picked up by both pages), keep only the earliest occurrence.

In [ ]:
KEY_COLS = ['Headword', 'Entry_Type', 'Parent_Lexeme', 'Sense_Number', 'Homonym_Number']

# Sort by page number so we keep the earliest page
df['_page_num'] = df['Source_Page'].str.extract(r'(\d+)').astype(int)
df_sorted = df.sort_values('_page_num')

dupes = df_sorted[df_sorted.duplicated(subset=KEY_COLS, keep=False)]
to_remove = df_sorted[df_sorted.duplicated(subset=KEY_COLS, keep='first')]

print(f'Found {len(to_remove)} cross-page duplicate rows to remove:')
if len(to_remove):
    display(to_remove[['Source_Page', 'Entry_Type', 'Headword', 'Sense_Number', 'Homonym_Number', 'Gloss']])


In [ ]:
# Apply deduplication and drop helper column
df_clean = df_sorted.drop_duplicates(subset=KEY_COLS, keep='first').drop(columns='_page_num').reset_index(drop=True)
print(f'Rows before: {len(df)} → after: {len(df_clean)}')

# Save polished TSV
df_clean.to_csv('../outputs/carolinian_polished.tsv', sep='	', index=False)
print('Saved → outputs/carolinian_polished.tsv')


### Resolve variant entries

Entries with "Variant of XXX" gloss get their fields filled in from the target headword.

In [ ]:
COPY_COLS = ['Gloss', 'Gloss_Secondary', 'POS', 'Usage_Note', 'Phonetic',
             'Cross_References', 'Examples', 'Example_Glosses',
             'Morphemic_Representation', 'Etymology', 'Inflection', 'Dialect']

# Identify variant entries
is_variant = (
    df_clean['Gloss'].str.strip().str.startswith('Variant of') |
    ((df_clean['Variant_Form'].str.strip() != '') & (df_clean['Gloss'].str.strip() == ''))
)

# Extract target headword from Gloss
df_clean['_target'] = df_clean['Gloss'].str.extract(r'Variant of (.+)', expand=False).str.strip()
# Fallback: use Variant_Form when gloss is empty
mask_empty = df_clean['_target'].isna() & (df_clean['Variant_Form'].str.strip() != '')
df_clean.loc[mask_empty, '_target'] = df_clean.loc[mask_empty, 'Variant_Form']

# Build lookup: headword -> first main entry row
lookup = df_clean[df_clean['Entry_Type'] == 'main'].drop_duplicates(subset='Headword').set_index('Headword')

resolved, not_found = [], []
for idx, row in df_clean[is_variant].iterrows():
    target = row['_target']
    if pd.isna(target) or target == '':
        continue
    if target in lookup.index:
        resolved.append((idx, target))
    else:
        not_found.append((row['Headword'], target, row['Source_Page']))

print(f'Resolvable: {len(resolved)} | Target not found: {len(not_found)}')
if not_found:
    print('Could not resolve (target not yet processed):')
    for hw, target, pg in not_found:
        print(f'  {hw} -> "{target}" ({pg})')


In [ ]:
# Copy fields from target into variant rows (only fills empty fields)
df_resolved = df_clean.copy()
for idx, target in resolved:
    for col in COPY_COLS:
        if df_resolved.at[idx, col].strip() == '':
            df_resolved.at[idx, col] = lookup.at[target, col]

df_resolved = df_resolved.drop(columns='_target')
print(f'Resolved {len(resolved)} variant entries')

# Save
df_resolved.to_csv('../outputs/carolinian_polished.tsv', sep='\t', index=False)
print('Saved -> outputs/carolinian_polished.tsv')
